<a href="https://colab.research.google.com/github/PSD-20/Portfolio-Optimization/blob/main/dynamic_portfolio_data_yfinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import random
import pandas as pd
import yfinance as yf


def prepare_portfolio_data(number_of_stocks, years, interval="1d", random_state=None):
    """
    Randomly selects stocks, downloads historical closing prices from Yahoo Finance,
    cleans the data, calculates returns, expected returns, and covariance matrix.

    Returns
    -------
    selected_stocks : list
    prices : pandas.DataFrame
    returns : pandas.DataFrame
    expected_returns : pandas.Series
    covariance_matrix : pandas.DataFrame
    """

    # --------------------------------------------------
    # STOCK UNIVERSE
    # --------------------------------------------------

    stock_universe = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
        "META", "TSLA", "AVGO", "JPM", "V",
        "MA", "WMT", "COST", "NFLX", "AMD",
        "ADBE", "CRM", "ORCL", "INTC", "QCOM",
        "CSCO", "IBM", "TXN", "AMAT", "MU",
        "PEP", "KO", "MCD", "SBUX", "NKE",
        "DIS", "HD", "LOW", "TGT", "CVX",
        "XOM", "COP", "BA", "CAT", "GE",
        "HON", "UPS", "FDX", "GS", "BAC",
        "MS", "AXP", "UNH", "JNJ", "PFE"
    ]

    # --------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------

    if not isinstance(number_of_stocks, int) or number_of_stocks <= 0:
        raise ValueError("number_of_stocks must be a positive integer.")

    if number_of_stocks > len(stock_universe):
        raise ValueError(
            f"Maximum number of stocks allowed is {len(stock_universe)}."
        )

    if not isinstance(years, int) or years <= 0:
        raise ValueError("years must be a positive integer.")

    if interval not in ["1d", "1wk"]:
        raise ValueError("interval must be either '1d' or '1wk'.")

    # --------------------------------------------------
    # RANDOM STOCK SELECTION
    # --------------------------------------------------

    rng = random.Random(random_state)

    selected_stocks = rng.sample(
        stock_universe,
        number_of_stocks
    )

    # --------------------------------------------------
    # DATE RANGE
    # --------------------------------------------------

    end_date = pd.Timestamp.today().normalize()
    start_date = end_date - pd.DateOffset(years=years)

    # --------------------------------------------------
    # DOWNLOAD DATA
    # --------------------------------------------------

    data = yf.download(
        tickers=selected_stocks,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        progress=False,
        group_by="column",
        threads=True
    )

    if data.empty:
        raise ValueError("Yahoo Finance returned no data.")

    # --------------------------------------------------
    # EXTRACT CLOSE PRICES
    # --------------------------------------------------

    if isinstance(data.columns, pd.MultiIndex):

        if "Close" in data.columns.get_level_values(0):
            prices = data["Close"].copy()

        elif "Close" in data.columns.get_level_values(1):
            prices = data.xs(
                "Close",
                axis=1,
                level=1
            ).copy()

        else:
            raise ValueError("Close prices not found.")

    else:

        if "Close" not in data.columns:
            raise ValueError("Close prices not found.")

        prices = data[["Close"]].copy()

        if number_of_stocks == 1:
            prices.columns = selected_stocks

    # --------------------------------------------------
    # CHECK TICKERS
    # --------------------------------------------------

    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    missing_stocks = [
        stock
        for stock in selected_stocks
        if stock not in prices.columns
    ]

    if missing_stocks:
        raise ValueError(
            f"Data missing for these stocks: {missing_stocks}"
        )

    # Preserve order
    prices = prices.reindex(columns=selected_stocks)

    # --------------------------------------------------
    # CLEAN PRICES
    # --------------------------------------------------

    prices = prices.apply(
        pd.to_numeric,
        errors="coerce"
    )

    prices = prices.dropna(how="any")

    if len(prices) < 2:
        raise ValueError("Not enough valid historical data.")

    # --------------------------------------------------
    # RETURNS
    # --------------------------------------------------

    returns = prices.pct_change().dropna()

    # --------------------------------------------------
    # EXPECTED RETURNS
    # --------------------------------------------------

    expected_returns = returns.mean()

    # --------------------------------------------------
    # COVARIANCE MATRIX
    # --------------------------------------------------

    covariance_matrix = returns.cov()

    # Preserve ordering
    expected_returns = expected_returns.reindex(selected_stocks)

    covariance_matrix = covariance_matrix.reindex(
        index=selected_stocks,
        columns=selected_stocks
    )

    # --------------------------------------------------
    # RETURN MODEL INPUT DATA
    # --------------------------------------------------

    return (
        selected_stocks,
        prices,
        returns,
        expected_returns,
        covariance_matrix
    )


# ======================================================
# USER INPUT
# ======================================================

number_of_stocks = int(
    input("Enter number of stocks you want: ")
)

years = int(
    input("Enter number of historical years: ")
)

interval = input(
    "Enter interval (1d for daily / 1wk for weekly): "
).strip()


# ======================================================
# RUN FUNCTION
# ======================================================

(
    selected_stocks,
    prices,
    returns,
    expected_returns,
    covariance_matrix
) = prepare_portfolio_data(
    number_of_stocks=number_of_stocks,
    years=years,
    interval=interval
)


# ======================================================
# OUTPUT
# ======================================================

print("\n==============================")
print("SELECTED STOCKS")
print("==============================")
print(selected_stocks)

print("\nNumber of stocks:")
print(len(selected_stocks))


print("\n==============================")
print("HISTORICAL PRICES")
print("==============================")
display(prices.head())


print("\n==============================")
print("HISTORICAL RETURNS")
print("==============================")
display(returns.head())


print("\n==============================")
print("EXPECTED RETURNS")
print("==============================")
display(expected_returns)


print("\n==============================")
print("COVARIANCE MATRIX")
print("==============================")
display(covariance_matrix)


print("\n==============================")
print("DIMENSIONS")
print("==============================")

print("Prices:", prices.shape)
print("Returns:", returns.shape)
print("Expected Returns:", expected_returns.shape)
print("Covariance Matrix:", covariance_matrix.shape)


print("\n==============================")
print("MODEL INPUT DATA READY")
print("==============================")

Enter number of stocks you want: 5
Enter number of historical years: 3
Enter interval (1d for daily / 1wk for weekly): 1d

SELECTED STOCKS
['COST', 'V', 'PEP', 'MSFT', 'AAPL']

Number of stocks:
5

HISTORICAL PRICES


Ticker,COST,V,PEP,MSFT,AAPL
Date,,,,,
2023-09-08,551.190002,247.289993,176.270004,334.269989,178.179993
2023-09-11,558.780029,247.220001,178.929993,337.940002,179.360001
2023-09-12,558.789978,247.300003,178.270004,331.769989,176.300003
2023-09-13,559.760010,247.830002,179.679993,336.059998,174.210007
2023-09-14,564.770020,241.500000,181.229996,338.700012,175.740005



HISTORICAL RETURNS


Ticker,COST,V,PEP,MSFT,AAPL
Date,,,,,
2023-09-11,0.013770,-0.000283,0.015090,0.010979,0.006623
2023-09-12,0.000018,0.000324,-0.003689,-0.018258,-0.017061
2023-09-13,0.001736,0.002143,0.007909,0.012931,-0.011855
2023-09-14,0.008950,-0.025542,0.008626,0.007856,0.008782
2023-09-15,-0.014891,-0.001781,-0.007670,-0.025037,-0.004154



EXPECTED RETURNS


,0
Ticker,
COST,0.000760
V,0.000637
PEP,-0.000250
MSFT,0.000672
AAPL,0.000922



COVARIANCE MATRIX


Ticker,COST,V,PEP,MSFT,AAPL
Ticker,,,,,
COST,0.000165,0.000057,0.000040,0.000040,0.000060
V,0.000057,0.000163,0.000030,0.000060,0.000073
PEP,0.000040,0.000030,0.000161,-0.000009,0.000034
MSFT,0.000040,0.000060,-0.000009,0.000274,0.000098
AAPL,0.000060,0.000073,0.000034,0.000098,0.000285



DIMENSIONS
Prices: (751, 5)
Returns: (750, 5)
Expected Returns: (5,)
Covariance Matrix: (5, 5)

MODEL INPUT DATA READY


NameError: name 'number_of_stocks' is not defined